# Quick Run: AU Corrections

Execute and visualize all four steps.

In [ ]:
import os, subprocess, sys
from pathlib import Path

repo = Path('/content/AIB')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/samiraghafarigousheh-sys/aib.git', str(repo)], check=True, capture_output=True)

os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib', 'pandas'], capture_output=True)
print('Setup complete')

In [ ]:
result = subprocess.run([sys.executable, 'examples/compare_au_corrections.py', '--through-step', '4'],
                       capture_output=True, text=True, timeout=3600)
print('Running harness...')
if result.returncode == 0:
    print('Completed successfully')
else:
    print(f'Error: {result.stderr[-200:]}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('results/au_corrections_summary/comparison.csv')
print(df.to_string(index=False))

baseline = df[df['State'] == 'Baseline'].iloc[0]['Total (kWh/m²)']
final = df.iloc[-1]['Total (kWh/m²)']
print(f'\nReduction: {baseline:.1f} -> {final:.1f} kWh/m² (−{100*(baseline-final)/baseline:.1f}%)')

In [ ]:
df_m = pd.read_csv('results/au_corrections_summary/comparison_by_metric.csv')
metrics = df_m['Metric'].tolist()
states = [c for c in df_m.columns if c != 'Metric']

fig, ax = plt.subplots(figsize=(14, 7))
x = range(len(metrics))
for i, s in enumerate(states):
    vals = pd.to_numeric(df_m[s], errors='coerce')
    ax.bar([j + 0.14*(i-len(states)/2) for j in x], vals, 0.14, label=s)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(ncol=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()